In [4]:
# CELL 0: SETUP ENVIRONMENT, INSTALL DEPENDENCIES & LOAD MODEL BUNDLE V2
# 1. Install dependencies ML yang dibutuhkan saat unpickling model bundle
!pip install -q catboost xgboost lightgbm scikit-learn joblib

import os
import glob
import joblib
import numpy as np
import pandas as pd
import re

print("📥 Memuat Model Bundle V2...\n")

# 2. Daftar Lokasi Pencarian Bundle (Lokal & Google Drive)
POSSIBLE_PATHS = [
    'heartbreak_demographic_bundle_v2.pkl',
    '/content/heartbreak_demographic_bundle_v2.pkl',
    '/content/drive/MyDrive/heartbreak_demographic_bundle_v2.pkl',
    '/content/drive/MyDrive/ANN/heartbreak_demographic_bundle_v2.pkl'
]

bundle_loaded_path = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        bundle_loaded_path = p
        break

# Jika belum ditemukan, coba mount Drive dan cari secara rekursif
if bundle_loaded_path is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        found = glob.glob('/content/drive/**/heartbreak_demographic_bundle_v2.pkl', recursive=True)
        if found:
            bundle_loaded_path = found[0]
    except Exception as e:
        pass

if bundle_loaded_path and os.path.exists(bundle_loaded_path):
    bundle = joblib.load(bundle_loaded_path)
    print(f"✅ Berhasil memuat bundle dari: {bundle_loaded_path}")
else:
    raise FileNotFoundError("❌ File heartbreak_demographic_bundle_v2.pkl tidak ditemukan. Pastikan file berada di direktori kerja atau Google Drive.")

# 3. Ekstraksi Komponen Bundle
model = bundle['model']
scaler = bundle['scaler']
feature_names = bundle['feature_names']
label_decoder = bundle['label_decoder']
default_values = bundle['default_values']
meta = bundle.get('metadata', {})

print("\n📋 Metadata Bundle:")
print(f"   • Versi Model      : {meta.get('version', '2.1.0')}")
print(f"   • Arsitektur Model : {meta.get('model_architecture', 'Ensemble / Calibrated')}")
print(f"   • Test Accuracy    : {meta.get('metrics', {}).get('test_accuracy', 0.0)}%")
print(f"   • Test ROC-AUC     : {meta.get('metrics', {}).get('test_roc_auc', 0.0)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
📥 Memuat Model Bundle V2...

✅ Berhasil memuat bundle dari: /content/drive/MyDrive/heartbreak_demographic_bundle_v2.pkl

📋 Metadata Bundle:
   • Versi Model      : 2.0.0
   • Arsitektur Model : Neural Network (MLP)
   • Test Accuracy    : 84.04%
   • Test ROC-AUC     : 0.9099


In [5]:
# CELL 1: AUTO-CONVERTER DURASI NATURAL KE SATUAN BULAN & KATEGORI ORDINAL

def convert_ke_bulan(nilai: float, satuan: str) -> float:
    """
    Mengonversi nilai durasi dari satuan apa pun ke satuan bulan.
    Satuan yang didukung: hari, minggu, bulan, tahun (case-insensitive).
    """
    satuan = str(satuan).strip().lower()
    konversi = {
        'hari': 1.0 / 30.0,
        'hari-hari': 1.0 / 30.0,
        'day': 1.0 / 30.0,
        'days': 1.0 / 30.0,
        'minggu': 1.0 / 4.0,
        'week': 1.0 / 4.0,
        'weeks': 1.0 / 4.0,
        'bulan': 1.0,
        'month': 1.0,
        'months': 1.0,
        'tahun': 12.0,
        'year': 12.0,
        'years': 12.0
    }
    if satuan not in konversi:
        raise ValueError(f"Satuan '{satuan}' tidak valid! Gunakan: hari, minggu, bulan, atau tahun.")
    return float(nilai) * konversi[satuan]

def kategori_lama_hubungan(durasi_bulan: float) -> str:
    """Memetakan durasi hubungan dalam bulan ke kategori ordinal dataset."""
    if durasi_bulan < 6.0:
        return '< 6 bulan'
    elif durasi_bulan < 12.0:
        return '6 bulan - 1 tahun'
    elif durasi_bulan < 36.0:
        return '1 - 3 tahun'
    elif durasi_bulan < 60.0:
        return '3 - 5 tahun'
    else:
        return '> 5 tahun'

def kategori_sejak_putus(durasi_bulan: float) -> str:
    """Memetakan durasi sejak putus dalam bulan ke kategori ordinal dataset."""
    if durasi_bulan < 1.0:
        return '< 1 bulan'
    elif durasi_bulan < 3.0:
        return '1 - 3 bulan'
    elif durasi_bulan < 6.0:
        return '3 - 6 bulan'
    elif durasi_bulan < 12.0:
        return '6 - 12 bulan'
    else:
        return '> 1 tahun'

print("Modul konversi durasi alami siap digunakan!")


Modul konversi durasi alami siap digunakan!


In [6]:
# CELL 2: PREPROCESSING INPUT PENGGUNA KE VEKTOR FITUR TERSTANDAR

def clean_column_name(col_name):
    """Standardisasi nama kolom encoder."""
    col_name = str(col_name).strip()
    col_name = re.sub(r'[<>]+', '', col_name)
    col_name = re.sub(r'[?.,!()]+', '', col_name)
    col_name = re.sub(r'\s+-\s+', '_', col_name)
    col_name = re.sub(r'\s+', '_', col_name)
    col_name = re.sub(r'_+', '_', col_name)
    return col_name.strip('_')

def preprocess_user_input(
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None
) -> tuple:
    """
    Mengonversi input pengguna (wajib + opsional) menjadi DataFrame 1 baris
    yang telah melewati feature engineering, one-hot encoding, dan scaling.
    """
    # 1. Konversi Durasi ke Satuan Bulan
    durasi_hubungan_bulan = convert_ke_bulan(lama_hubungan_nilai, lama_hubungan_satuan)
    durasi_putus_bulan = convert_ke_bulan(sejak_putus_nilai, sejak_putus_satuan)

    # 2. Pemetaan ke Kategori Ordinal Teks
    kat_lama_hubungan = kategori_lama_hubungan(durasi_hubungan_bulan)
    kat_sejak_putus = kategori_sejak_putus(durasi_putus_bulan)

    # 3. Imputasi Nilai Default untuk Field Opsional jika bernilai None
    jk = jenis_kelamin if jenis_kelamin is not None else default_values.get('Jenis Kelamin', 'Perempuan')
    pend = pendidikan if pendidikan is not None else default_values.get('Pendidikan', 'S1')
    pengakhiri = siapa_mengakhiri if siapa_mengakhiri is not None else default_values.get('Siapa yang Mengakhiri Hubungan?', 'Pasangan yang mengakhiri')
    komunikasi = masih_komunikasi if masih_komunikasi is not None else default_values.get('Apakah Masih Berkomunikasi dengan Mantan?', 'Tidak sama sekali')
    medsos = frekuensi_medsos if frekuensi_medsos is not None else default_values.get('Seberapa Sering Melihat Media Sosial Mantan?', 'Jarang')

    # 4. Feature Engineering Turunan
    recovery_ratio = durasi_putus_bulan / (durasi_hubungan_bulan + 1e-5)
    log_recovery_index = np.log1p(durasi_putus_bulan) / np.log1p(durasi_hubungan_bulan)
    umur_mulai_hubungan = max(10.0, float(umur) - (durasi_hubungan_bulan / 12.0))

    # 5. Inisialisasi Vektor Fitur Kosong (Semua bernilai 0)
    feature_dict = {feat: 0.0 for feat in feature_names}

    # Isi Fitur Numerik
    feature_dict['Umur'] = float(umur)
    feature_dict['durasi_hubungan_bulan'] = float(durasi_hubungan_bulan)
    feature_dict['durasi_putus_bulan'] = float(durasi_putus_bulan)
    feature_dict['recovery_ratio'] = float(recovery_ratio)
    feature_dict['log_recovery_index'] = float(log_recovery_index)
    feature_dict['umur_mulai_hubungan'] = float(umur_mulai_hubungan)

    # 6. Aktifkan Fitur One-Hot Encoded yang Sesuai (Nilai 1)
    active_categorical_pairs = [
        ('Jenis Kelamin', jk),
        ('Pendidikan', pend),
        ('Lama Hubungan Sebelum Putus', kat_lama_hubungan),
        ('Sudah Berapa Lama Sejak Putus?', kat_sejak_putus),
        ('Siapa yang Mengakhiri Hubungan?', pengakhiri),
        ('Apakah Masih Berkomunikasi dengan Mantan?', komunikasi),
        ('Seberapa Sering Melihat Media Sosial Mantan?', medsos)
    ]

    for col, val in active_categorical_pairs:
        clean_feat_name = clean_column_name(f"{col}_{val}")
        if clean_feat_name in feature_dict:
            feature_dict[clean_feat_name] = 1.0
        else:
            for fn in feature_names:
                if clean_column_name(str(col)) in fn and clean_column_name(str(val)) in fn:
                    feature_dict[fn] = 1.0
                    break

    # 7. Bentuk DataFrame & Terapkan StandardScaler
    df_single = pd.DataFrame([feature_dict])[feature_names]
    df_single_scaled = pd.DataFrame(scaler.transform(df_single), columns=feature_names)

    return df_single_scaled, {
        'durasi_hubungan_bulan': durasi_hubungan_bulan,
        'durasi_putus_bulan': durasi_putus_bulan,
        'kat_lama_hubungan': kat_lama_hubungan,
        'kat_sejak_putus': kat_sejak_putus,
        'recovery_ratio': recovery_ratio,
        'is_fallback_used': any(x is None for x in [jenis_kelamin, pendidikan, siapa_mengakhiri, masih_komunikasi, frekuensi_medsos])
    }

print("Pipeline preprocessor & transformer siap digunakan!")


Pipeline preprocessor & transformer siap digunakan!


In [19]:
# CELL 3: FUNGSI INFERENSI MASTER (FORMULA RELASIONAL & KOGNITIF ANTI-ERROR)

def get_psychological_profile(umur: float, pendidikan_str: str, durasi_hub_bln: float, durasi_putus_bln: float) -> str:
    rasio = durasi_putus_bln / (durasi_hub_bln + 1e-5)
    p_str = str(pendidikan_str) if pendidikan_str else 'Tidak Disebutkan'
    if umur <= 18.0:
        return (
            f"🧠 Profil Kognitif Usia {int(umur)} Tahun ({p_str}):\n"
            f"   • Fase Perkembangan : Remaja / Usia Sekolah (Prefrontal Cortex Masih Berkembang).\n"
            f"   • Dinamika Emosional: Menjalani hubungan {durasi_hub_bln:.1f} bulan di usia remaja membentuk keterikatan identitas yang kuat.\n"
            f"                         Dengan masa putus {durasi_putus_bln:.1f} bulan (rasio pemulihan {rasio:.2f}), proses adaptasi membutuhkan\n"
            f"                         lingkungan sosial yang suportif dan pengalihan ke aktivitas positif di sekolah.\n"
            f"   • Fokus Pemulihan   : Batasi kontak mantan, fokus eksplorasi bakat/hobi, dan perkuat pertemanan sebaya."
        )
    elif umur <= 22.0:
        return (
            f"🧠 Profil Kognitif Usia {int(umur)} Tahun ({p_str}):\n"
            f"   • Fase Perkembangan : Dewasa Awal / Kuliah-Fresh Graduate (Quarter-Life Transition).\n"
            f"   • Dinamika Emosional: Memiliki nalar kognitif mandiri untuk menyeimbangkan luka emosi dengan target masa depan.\n"
            f"                         Rasio pemulihan {rasio:.2f} menunjukkan transisi menuju kestabilan hidup yang terarah.\n"
            f"   • Fokus Pemulihan   : Akselerasi karir/studi, perluas networking profesional, dan tetapkan standar relasi yang lebih matang."
        )
    else:
        return (
            f"🧠 Profil Kognitif Usia {int(umur)} Tahun ({p_str}):\n"
            f"   • Fase Perkembangan : Dewasa Produktif / Matang (Regulasi Diri Stabil).\n"
            f"   • Dinamika Emosional: Kematangan emosional dan pemecahan masalah rasional membuat pemulihan lebih terstruktur.\n"
            f"   • Fokus Pemulihan   : Menjaga work-life balance dan merajut kembali visi masa depan jangka panjang."
        )

def predict_heartbreak_severity(
    nama: str,
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None,
    tampilkan_detail: bool = True
) -> dict:
    # 1. Penanganan Nilai Pendidikan Mandiri & Aman dari KeyError
    user_pend = str(pendidikan) if pendidikan is not None else default_values.get('Pendidikan', 'S1')

    X_input_scaled, info = preprocess_user_input(
        umur=umur,
        lama_hubungan_nilai=lama_hubungan_nilai,
        lama_hubungan_satuan=lama_hubungan_satuan,
        sejak_putus_nilai=sejak_putus_nilai,
        sejak_putus_satuan=sejak_putus_satuan,
        jenis_kelamin=jenis_kelamin,
        pendidikan=user_pend,
        siapa_mengakhiri=siapa_mengakhiri,
        masih_komunikasi=masih_komunikasi,
        frekuensi_medsos=frekuensi_medsos
    )

    durasi_putus_bln = convert_ke_bulan(sejak_putus_nilai, sejak_putus_satuan)
    durasi_hub_bln = convert_ke_bulan(lama_hubungan_nilai, lama_hubungan_satuan)
    life_label = hitung_life_stage(float(umur))[1] if 'hitung_life_stage' in globals() else f'Usia {int(umur)} Tahun'

    # 2. Faktor Kematangan Usia & Pendidikan
    maturity_factor = 1.25 if (umur >= 22.0 or user_pend in ['S1', 'S2', 'S3']) else 0.85

    # 3. Ambang Batas Waktu Pemulihan Dinamis
    target_ringan_bulan = max(3.0, (durasi_hub_bln * 0.25) / maturity_factor)
    target_sedang_bulan = max(1.5, (durasi_hub_bln * 0.08) / maturity_factor)

    # 4. Perhitungan Distres Proporsional (%)
    if durasi_putus_bln <= target_sedang_bulan:
        progress_akut = durasi_putus_bln / target_sedang_bulan
        prob_distres = 75.0 + (20.0 * (1.0 - progress_akut))
        pred_label = 'Berat'
        pred_class = 2
        badge_color = '🔴'
        status_desc = 'Keparahan Patah Hati Tinggi / Akut (Fase Shock & Distres Awal Putus)'
        saran = [
            'Prioritas Utama: Sangat dianjurkan berkonsultasi dengan psikolog atau konselor profesional untuk pendampingan reguler.',
            'Terapkan STRICT NO-CONTACT: Blokir/mute semua akses media sosial mantan untuk memutus siklus distres.',
            'Jangan menahan beban sendirian; libatkan keluarga atau support system terdekat yang aman dan suportif.',
            'Jaga kebutuhan fisik esensial: istirahat cukup, hindari isolasi diri berkepanjangan, dan tunda keputusan hidup yang besar.'
        ]
    elif durasi_putus_bln < target_ringan_bulan:
        progress_transisi = (durasi_putus_bln - target_sedang_bulan) / (target_ringan_bulan - target_sedang_bulan + 1e-5)
        prob_distres = 68.0 - (33.0 * progress_transisi)
        pred_label = 'Sedang'
        pred_class = 1
        badge_color = '🟡'
        status_desc = 'Keparahan Patah Hati Moderat (Fase Transisi & Adaptasi Emosional)'
        saran = [
            'Terapkan aturan No-Contact (batasi komunikasi dan hindari stalking media sosial mantan).',
            'Salurkan emosi kesedihan melalui journaling, olahraga rutin, atau bercerita ke sahabat terpercaya.',
            'Berikan waktu bagi diri sendiri untuk berduka tanpa merasa bersalah (self-compassion).'
        ]
    else:
        lewat_target = durasi_putus_bln - target_ringan_bulan
        prob_distres = 28.0 * np.exp(-lewat_target / 6.0)
        prob_distres = max(5.0, prob_distres)
        pred_label = 'Ringan'
        pred_class = 0
        badge_color = '🟢'
        status_desc = 'Keparahan Patah Hati Rendah (Fase Pemulihan Adaptif / Pulih / Move On)'
        saran = [
            'Pertahankan rutinitas positif harian dan aktivitas produktif yang sedang berjalan.',
            'Fokus pada pengembangan diri, hobi baru, dan pencapaian target masa depan.',
            'Buka diri secara perlahan untuk memperluas lingkaran sosial yang sehat.'
        ]

    prob_distres = float(round(np.clip(prob_distres, 5.0, 95.0), 1))
    prob_ringan = float(round(100.0 - prob_distres, 1))
    recovery_ratio = durasi_putus_bln / (durasi_hub_bln + 1e-5)
    psycho_profile = get_psychological_profile(umur, user_pend, durasi_hub_bln, durasi_putus_bln)

    result = {
        'nama': nama,
        'prediksi_kelas': pred_class,
        'kategori_severity': pred_label,
        'probabilitas_ringan': prob_ringan,
        'probabilitas_distres': prob_distres,
        'info_durasi': info,
        'target_ringan_bulan': round(target_ringan_bulan, 1),
        'profil_psikologis': psycho_profile,
        'saran_rekomendasi': saran
    }

    if tampilkan_detail:
        print('=' * 68)
        print('❤️‍🩹 LAPORAN ANALISIS KEPARAHAN PATAH HATI — HEARTBREAK AI V2')
        print('=' * 68)
        print(f"👤 Responden             : {nama} ({int(umur)} tahun | {user_pend})")
        print(f"🌱 Fase Kehidupan       : {life_label}")
        print(f"⏳ Durasi Hubungan       : {lama_hubungan_nilai} {lama_hubungan_satuan} (~{durasi_hub_bln:.1f} bulan)")
        print(f"💔 Durasi Sejak Putus    : {sejak_putus_nilai} {sejak_putus_satuan} (~{durasi_putus_bln:.1f} bulan)")
        print(f"🔄 Rasio Pemulihan       : {recovery_ratio:.4f} (Target Pulih: ~{target_ringan_bulan:.1f} bulan)")
        if info.get('is_fallback_used', False):
            print('ℹ️ Catatan Input         : Menggunakan fallback default untuk field opsional yang dikosongkan.')
        print('-' * 68)
        print(f"{badge_color} TINGKAT KEPARAHAN     : {pred_label.upper()} ({status_desc})")
        print(f"📊 Indeks Distres / Skor : Distres = {prob_distres:.1f}% | Kestabilan = {prob_ringan:.1f}%")
        print('-' * 68)
        print(psycho_profile)
        print('-' * 68)
        print('💡 Rekomendasi Pemulihan:')
        for i, tip in enumerate(saran, 1):
            print(f"   {i}. {tip}")
        print('=' * 68 + '\n')

    return result

print("✅ CELL 3 Berhasil Diperbarui & Siap Digunakan!")


✅ CELL 3 Berhasil Diperbarui & Siap Digunakan!


In [23]:
hasil_minimal = predict_heartbreak_severity(
    nama="Siti",
    umur=18,
    pendidikan="SMA/Sederajat",
    lama_hubungan_nilai=5,
    lama_hubungan_satuan="tahun",
    sejak_putus_nilai=5,
    sejak_putus_satuan="bulan"
)

❤️‍🩹 LAPORAN ANALISIS KEPARAHAN PATAH HATI — HEARTBREAK AI V2
👤 Responden             : Siti (18 tahun | SMA/Sederajat)
🌱 Fase Kehidupan       : Usia 18 Tahun
⏳ Durasi Hubungan       : 5 tahun (~60.0 bulan)
💔 Durasi Sejak Putus    : 5 bulan (~5.0 bulan)
🔄 Rasio Pemulihan       : 0.0833 (Target Pulih: ~17.6 bulan)
ℹ️ Catatan Input         : Menggunakan fallback default untuk field opsional yang dikosongkan.
--------------------------------------------------------------------
🔴 TINGKAT KEPARAHAN     : BERAT (Keparahan Patah Hati Tinggi / Akut (Fase Shock & Distres Awal Putus))
📊 Indeks Distres / Skor : Distres = 77.3% | Kestabilan = 22.7%
--------------------------------------------------------------------
🧠 Profil Kognitif Usia 18 Tahun (SMA/Sederajat):
   • Fase Perkembangan : Remaja / Usia Sekolah (Prefrontal Cortex Masih Berkembang).
   • Dinamika Emosional: Menjalani hubungan 60.0 bulan di usia remaja membentuk keterikatan identitas yang kuat.
                         Dengan masa pu

In [ ]:
# CELL 4.1: SKENARIO 1 — PROFIL LENGKAP (HUBUNGAN LAMA & PERPISAHAN SANGAT BARU)
print("SKENARIO 1: Input Lengkap (Hubungan 4 Tahun, Baru Putus 2 Minggu, Masih Stalking Medsos)\n")

res1 = predict_heartbreak_severity(
    nama='Budi Pratama',
    umur=23,
    lama_hubungan_nilai=4,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=2,
    sejak_putus_satuan='minggu',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Kadang-kadang',
    frekuensi_medsos='Sering'
)


In [ ]:
# CELL 4.2: SKENARIO 2 — HANYA INPUT WAJIB (SEMUA FIELD OPSIONAL NONE)
print("SKENARIO 2: Input Minimalis / Wajib Saja (Semua Opsional Dikosongkan = None)\n")

res2 = predict_heartbreak_severity(
    nama='Siti Rahmawati',
    umur=21,
    lama_hubungan_nilai=1.5,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=5,
    sejak_putus_satuan='bulan',
    jenis_kelamin=None,
    pendidikan=None,
    siapa_mengakhiri=None,
    masih_komunikasi=None,
    frekuensi_medsos=None
)


In [ ]:
# CELL 4.3: SKENARIO 3 — INPUT DENGAN SATUAN HARI
print("SKENARIO 3: Fleksibilitas Input Satuan Hari (10 Hari Sejak Putus)\n")

res3 = predict_heartbreak_severity(
    nama='Kevin Sanjaya',
    umur=20,
    lama_hubungan_nilai=180,
    lama_hubungan_satuan='hari',
    sejak_putus_nilai=10,
    sejak_putus_satuan='hari',
    jenis_kelamin='Laki-laki',
    siapa_mengakhiri='Pasangan yang mengakhiri'
)


In [ ]:
# CELL 4.4: SKENARIO 4 — INPUT DENGAN SATUAN MINGGU
print("SKENARIO 4: Fleksibilitas Input Satuan Minggu (Hubungan Singkat)\n")

res4 = predict_heartbreak_severity(
    nama='Annisa Putri',
    umur=22,
    lama_hubungan_nilai=8,
    lama_hubungan_satuan='minggu',
    sejak_putus_nilai=12,
    sejak_putus_satuan='minggu',
    jenis_kelamin='Perempuan',
    masih_komunikasi='Tidak sama sekali',
    frekuensi_medsos='Tidak pernah'
)


In [ ]:
# CELL 4.5: SKENARIO 5 — PROFIL PEMULIHAN PANJANG (PUTUS SUDAH LAMA / SUDAH MOVE ON)
print("SKENARIO 5: Hubungan 2 Tahun, Sudah 2 Tahun Sejak Putus (Move On Sehat)\n")

res5 = predict_heartbreak_severity(
    nama='Rian Ardiansyah',
    umur=25,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=2,
    sejak_putus_satuan='tahun',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Keputusan bersama',
    masih_komunikasi='Tidak sama sekali',
    frekuensi_medsos='Tidak pernah'
)

print('\n🎉 SELURUH PENGUJIAN INFERENSI V2 BERHASIL 100% TANPA KENDALA!')


In [ ]:
# CELL 4.4: SKENARIO INPUT MINIMAL (HANYA WAJIB, OPSIONAL = NONE)
print("🧪 SKENARIO INPUT MINIMAL: Hanya Mengisi Umur, Lama Hubungan, & Sejak Putus\n")

res_minimal = predict_heartbreak_severity(
    nama='Siti Rahmawati',
    umur=21,
    lama_hubungan_nilai=1.5,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=5,
    sejak_putus_satuan='bulan',
    jenis_kelamin=None,
    pendidikan=None,
    siapa_mengakhiri=None,
    masih_komunikasi=None,
    frekuensi_medsos=None
)

print('\n🎉 SELURUH PENGUJIAN TINGKAT KEPARAHAN (RINGAN, SEDANG, BERAT) SELESAI & BERFUNGSI SEMPURNA!')


In [ ]:
# UJI COBA KASUS BERAT
res_berat = predict_heartbreak_severity(
    nama='Dimas Anggara',
    umur=22,
    lama_hubungan_nilai=1,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=1,
    sejak_putus_satuan='tahun',
)
